# Quickstart: union-selection rectangular-sieve fit

This notebook generates one bounded-Gaussian catalogue, retains Regions A--C, fits the conditional-likelihood sieve, and inspects the fitted CDF and KKT diagnostics. It is a worked example for the repository, not an additional experiment in the paper.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from union_selection_sieve import (
    GAUSSIAN_LOWER, GAUSSIAN_UPPER, GAUSSIAN_Y_MASS, GAUSSIAN_Y_SUPPORT,
    FitConfig, TruncatedBivariateNormalTruth, fit_catalogue,
    simulate_gaussian_catalogue, uniform_grid,
)

In [ ]:
rho = 0.55
latent, observed, simulation_info = simulate_gaussian_catalogue(1200, 20260912, rho, GAUSSIAN_LOWER, GAUSSIAN_UPPER, GAUSSIAN_Y_SUPPORT, GAUSSIAN_Y_MASS)
simulation_info, observed["region"].value_counts().sort_index()

In [ ]:
grid = uniform_grid(12, GAUSSIAN_LOWER, GAUSSIAN_UPPER)
estimate = fit_catalogue(observed, grid, config=FitConfig())
estimate.summary()

In [ ]:
cdf = estimate.cdf_table(query_grid_size=41)
truth = TruncatedBivariateNormalTruth(rho)
cdf["cdf_truth"] = [truth.cdf(x1, x2) for x1, x2 in zip(cdf.x1, cdf.x2)]
cdf["error"] = cdf["cdf_equal_split"] - cdf["cdf_truth"]
float(np.sqrt(np.mean(cdf["error"] ** 2)))

In [ ]:
pivot = cdf.pivot(index="x2", columns="x1", values="cdf_equal_split")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(
    pivot.to_numpy(), origin="lower", aspect="auto",
    extent=[cdf.x1.min(), cdf.x1.max(), cdf.x2.min(), cdf.x2.max()]
)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
fig.colorbar(image, ax=ax, label=r"$\widehat F(x_1,x_2)$")
plt.show()

In [ ]:
outdir = Path("../outputs/notebook_demo")
estimate.write(outdir)
observed.to_csv(outdir / "observed_ABC_catalogue.csv", index=False)
outdir.resolve()